In [1]:
from deepanc.common.utils import loading_paths_from_MAT, nmse, apply_impulse_response
from deepanc.inference.antinoise_generator import Deep_ANC_controller
import torch
import numpy as np
import math
import soundfile as sf
from construct_dataset_from_freesound.resample import resampler

In [ ]:
from pyprojroot import here
PROJECT_ROOT = here()
MODEL_DIR = PROJECT_ROOT / "models"
PATHS_DIR = PROJECT_ROOT / "rirs"
NOISE_DIR = PROJECT_ROOT / "NOISEX92"
FIG_DIR = PROJECT_ROOT / "figs"
FIG_DIR.mkdir(parents=True, exist_ok=True)
print(PROJECT_ROOT)
print(MODEL_DIR)
print(PATHS_DIR)
print(NOISE_DIR)

/home/rkobayashi/anc/deepanc
/home/rkobayashi/anc/deepanc/models
/home/rkobayashi/anc/deepanc/rirs
/home/rkobayashi/anc/deepanc/NOISEX92


In [3]:
fs = 16000
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
delay = 1
rir = [f"rt60_{i}s" for i in [0.15, 0.175, 0.2, 0.225, 0.25]][2]
eta = torch.tensor([math.sqrt(0.1), math.sqrt(1), math.sqrt(10), torch.finfo(torch.float32).max], dtype=torch.float32)[3]
etas = torch.tensor([math.sqrt(0.1), math.sqrt(0.5), torch.finfo(torch.float32).max], dtype=torch.float32)
model_name = f"noise_fullband_delay_{delay}.pth"

In [8]:
Pri_path, Sec_path = torch.tensor(loading_paths_from_MAT(PATHS_DIR, rir, "Primary_path.mat", "Secondary_path.mat", )).unsqueeze(1)

# sound_names = [ sound_name.name + "/signal.wav" for sound_name in NOISE_DIR.iterdir() if sound_name.is_dir()]
sound_names = [ sound_name + "/signal.wav" for sound_name in ["destroyerengine", "factory1", "factory2", "babble"]]
for sound_index in range(len(sound_names)):
    for eta in etas:
        sound_name = sound_names[sound_index]
        print(f"{sound_name.split('/')[0]}, eta={eta}")
        waveform, sample_rate = sf.read(NOISE_DIR/sound_name, dtype="float32", )
        waveform = torch.from_numpy(resampler(waveform, sample_rate, fs))

        Dis, Ref = apply_impulse_response(waveform, Pri_path), waveform
        Time = np.arange(Dis.shape[1]) / fs

        controller = Deep_ANC_controller(MODEL_DIR/model_name, device, fs, delay, eta, Sec_path)
        Error = controller.noise_cancellation(Dis, Ref)

        print(f"  NMSE: {nmse(Error.detach(), Dis.detach())}")

    # Dis, Error = Dis.tolist()[0], Error.tolist()[0]

destroyerengine, eta=0.3162277638912201


/home/rkobayashi/anc/deepanc/src/deepanc/common/utils.py:28: UserWarning: The length of signal is shorter than the length parameter. Result is being padded with zeros in the tail. Please check your center and hop_length settings. (Triggered internally at ../aten/src/ATen/native/SpectralOps.cpp:1166.)
  wave = torch.istft(stft_complex, N_FFT, HOP_LENGTH, window=window, center=False, length=length)


  NMSE: -6.058398246765137
destroyerengine, eta=0.7071067690849304
  NMSE: -6.112436771392822
destroyerengine, eta=3.4028234663852886e+38
  NMSE: -6.125551223754883
factory1, eta=0.3162277638912201
  NMSE: -4.280976295471191
factory1, eta=0.7071067690849304
  NMSE: -4.296993255615234
factory1, eta=3.4028234663852886e+38
  NMSE: -4.3009467124938965
factory2, eta=0.3162277638912201
  NMSE: -4.024674415588379
factory2, eta=0.7071067690849304
  NMSE: -4.211753845214844
factory2, eta=3.4028234663852886e+38
  NMSE: -4.255362510681152
babble, eta=0.3162277638912201
  NMSE: -5.657471656799316
babble, eta=0.7071067690849304
  NMSE: -5.70619010925293
babble, eta=3.4028234663852886e+38
  NMSE: -5.714747905731201


In [9]:
model_name = f"freesound_delay_{delay}.pth"
Pri_path, Sec_path = torch.tensor(loading_paths_from_MAT(PATHS_DIR, rir, "Primary_path.mat", "Secondary_path.mat", )).unsqueeze(1)

# sound_names = [ sound_name.name + "/signal.wav" for sound_name in NOISE_DIR.iterdir() if sound_name.is_dir()]
sound_names = [ sound_name + "/signal.wav" for sound_name in ["destroyerengine", "factory1", "factory2", "babble"]]
for sound_index in range(len(sound_names)):
    for eta in etas:
        sound_name = sound_names[sound_index]
        print(f"{sound_name.split('/')[0]}, eta={eta}")
        waveform, sample_rate = sf.read(NOISE_DIR/sound_name, dtype="float32", )
        waveform = torch.from_numpy(resampler(waveform, sample_rate, fs))

        Dis, Ref = apply_impulse_response(waveform, Pri_path), waveform
        Time = np.arange(Dis.shape[1]) / fs

        controller = Deep_ANC_controller(MODEL_DIR/model_name, device, fs, delay, eta, Sec_path)
        Error = controller.noise_cancellation(Dis, Ref)

        print(f"  NMSE: {nmse(Error.detach(), Dis.detach())}")

    # Dis, Error = Dis.tolist()[0], Error.tolist()[0]

destroyerengine, eta=0.3162277638912201
  NMSE: -8.779898643493652
destroyerengine, eta=0.7071067690849304
  NMSE: -8.928437232971191
destroyerengine, eta=3.4028234663852886e+38
  NMSE: -8.96292781829834
factory1, eta=0.3162277638912201
  NMSE: -5.624102592468262
factory1, eta=0.7071067690849304
  NMSE: -5.659031867980957
factory1, eta=3.4028234663852886e+38
  NMSE: -5.667205810546875
factory2, eta=0.3162277638912201
  NMSE: -4.737865447998047
factory2, eta=0.7071067690849304
  NMSE: -5.031249523162842
factory2, eta=3.4028234663852886e+38
  NMSE: -5.095808506011963
babble, eta=0.3162277638912201
  NMSE: -8.191036224365234
babble, eta=0.7071067690849304
  NMSE: -8.323930740356445
babble, eta=3.4028234663852886e+38
  NMSE: -8.350883483886719


In [ ]:
# import matplotlib.pyplot as plt
# plt.title('The DeepANC Algorithm')
# plt.plot(Time, Dis, color='blue', label='ANC off')
# plt.plot(Time, Error, color='orange', label='ANC on')
# plt.ylabel('Magnitude')
# plt.xlabel('Time (seconds)')
# plt.legend()
# plt.grid()
# plt.savefig(FIG_DIR/'DeepANC.png', dpi=600, bbox_inches='tight', pad_inches=0)
# plt.show()